# NSE Stock Analysis - Complete Version

Fetches 1 year OHLCV data, generates analysis reports, charts, and sends email

## 1. Imports & Configuration

In [ ]:
import sys
import io

# Fix Windows console encoding
if sys.platform == 'win32':
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
    sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding='utf-8')

import pandas as pd
import numpy as np
from datetime import datetime
import datetime as dt
import os
import time
import requests
import concurrent.futures
from nselib import capital_market
import nselib.capital_market.capital_market_data as _cm_data
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage

print('Imports OK')

In [ ]:
# Email configuration
EMAIL_FROM = os.environ.get('EMAIL_FROM', 'peri47.study@gmail.com')
EMAIL_TO = os.environ.get('EMAIL_TO', 'peri47.study@gmail.com')
EMAIL_PASS = os.environ.get('EMAIL_PASS', 'fhih lsqp wanu leic')

# Load stock list
with open('stocks.txt', 'r') as f:
    STOCK_LIST = [line.strip() for line in f if line.strip() and not line.startswith('#')]

MAX_WORKERS = 5
DAYS = 365
OUTPUT_DIR = 'dump'

print(f'{len(STOCK_LIST)} stocks loaded from stocks.txt')

## 2. Helper Functions

In [ ]:
def compute_period_hl(df):
    """Calculate high/low for various periods"""
    df = df.sort_values('date').reset_index(drop=True)
    n = len(df)
    
    def window_hl(days):
        rows = df.tail(days)
        return rows['high'].max(), rows['low'].min()
    
    h52, l52 = window_hl(min(252, n))
    h26, l26 = window_hl(min(126, n))
    h4, l4 = window_hl(min(20, n))
    h1, l1 = window_hl(min(5, n))
    current = df['close'].iloc[-1]
    rng = h52 - l52
    pos = (current - l52) / rng * 100 if rng > 0 else float('nan')
    
    return {
        'Current_Price': round(current, 2),
        '52W_High': round(h52, 2), '52W_Low': round(l52, 2),
        '26W_High': round(h26, 2), '26W_Low': round(l26, 2),
        '4W_High': round(h4, 2), '4W_Low': round(l4, 2),
        '1W_High': round(h1, 2), '1W_Low': round(l1, 2),
        '52W_Position': round(pos, 2) if pos == pos else float('nan'),
    }

def compute_sma_crossovers(df):
    """Calculate SMA crossover signals"""
    df = df.sort_values('date').reset_index(drop=True)
    n = len(df)
    close = df['close']
    
    sma = {p: close.rolling(p).mean() if n >= p else pd.Series([float('nan')]*n, index=df.index)
           for p in [5, 10, 20, 50, 100, 200]}
    
    results = []
    for label, short_p, long_p in [('200/20', 20, 200), ('100/10', 10, 100), ('50/5', 5, 50)]:
        s, l = sma[short_p], sma[long_p]
        if s.isna().all() or l.isna().all():
            continue
        
        cross = (s > l) & (s.shift(1) <= l.shift(1))
        last5_cross = cross.iloc[-5:].any() if n >= 5 else False
        cross_dates = df['date'][cross]
        last_cross_date = cross_dates.iloc[-1].strftime('%Y-%m-%d') if not cross_dates.empty and last5_cross else None
        
        s_last, l_last = s.iloc[-1], l.iloc[-1]
        if pd.isna(s_last) or pd.isna(l_last):
            continue
        
        cross_pct = (s_last - l_last) / l_last * 100
        results.append({
            'cross_type': label,
            'crossed_last_5d': bool(last5_cross),
            'last_cross_date': last_cross_date,
            'nearing': bool(-1.0 <= cross_pct <= 0),
            'cross_pct': round(cross_pct, 3),
        })
    
    return results

def compute_macd(df):
    """Calculate MACD indicators"""
    df = df.sort_values('date').reset_index(drop=True)
    close = df['close']
    
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    signal_line = macd_line.ewm(span=9, adjust=False).mean()
    histogram = macd_line - signal_line
    
    n = len(df)
    cross = (macd_line > signal_line) & (macd_line.shift(1) <= signal_line.shift(1))
    bullish_cross = cross.iloc[-5:].any() if n >= 5 else False
    above_zero = bool(macd_line.iloc[-1] > 0)
    
    hist_inc = False
    if n >= 3:
        h = histogram.iloc[-3:].values
        hist_inc = bool(h[1] > h[0] and h[2] > h[1])
    
    score = int(bullish_cross) + int(above_zero) + int(hist_inc)
    
    return {
        'MACD': round(macd_line.iloc[-1], 4),
        'Signal': round(signal_line.iloc[-1], 4),
        'Histogram': round(histogram.iloc[-1], 4),
        'Bullish_Cross': bool(bullish_cross),
        'Above_Zero': above_zero,
        'Hist_Increasing': hist_inc,
        'MACD_Score': score,
    }

def compute_period_returns(df):
    """Calculate percentage returns over multiple periods"""
    df = df.sort_values('date').reset_index(drop=True)
    n = len(df)
    
    if n < 2:
        return None
    
    current_price = df['close'].iloc[-1]
    
    def calc_return(days_back):
        if n <= days_back:
            return float('nan')
        past_price = df['close'].iloc[-(days_back + 1)]
        if past_price == 0:
            return float('nan')
        return ((current_price - past_price) / past_price) * 100
    
    return {
        'Current_Price': round(current_price, 2),
        '2D_%': round(calc_return(2), 2),
        '5D_%': round(calc_return(5), 2),
        '10D_%': round(calc_return(10), 2),
        '1M_%': round(calc_return(21), 2),
        '3M_%': round(calc_return(63), 2),
        '6M_%': round(calc_return(126), 2),
        '1Y_%': round(calc_return(252), 2),
    }

print('Helper functions defined')

## 3. Fetch Data

In [ ]:
end_date = dt.date.today().strftime('%d-%m-%Y')
start_date = (dt.date.today() - dt.timedelta(days=DAYS)).strftime('%d-%m-%Y')
print(f'Date range: {start_date} to {end_date}')

# Initialize session
print('Initialising NSE session...')
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': '*/*',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'X-Requested-With': 'XMLHttpRequest',
    'Referer': 'https://www.nseindia.com/',
})

r0 = session.get('https://www.nseindia.com/', timeout=30)
print(f'  Session: HTTP {r0.status_code}')
time.sleep(2)

def shared_urlfetch(url, origin_url='http://nseindia.com'):
    return session.get(url, timeout=30)

_cm_data.nse_urlfetch = shared_urlfetch

In [ ]:
def fetch_one(sym):
    df = capital_market.price_volume_data(symbol=sym, from_date=start_date, to_date=end_date)
    for col in ['HighPrice', 'LowPrice', 'OpenPrice', 'ClosePrice', 'TotalTradedQuantity']:
        df[col] = df[col].astype(str).str.replace(',', '', regex=False).astype(float)
    df = df.rename(columns={
        'Symbol': 'symbol', 'Date': 'date',
        'HighPrice': 'high', 'LowPrice': 'low',
        'OpenPrice': 'open', 'ClosePrice': 'close',
        'TotalTradedQuantity': 'volume'
    })
    df['symbol'] = sym
    df['date'] = pd.to_datetime(df['date'], format='%d-%b-%Y')
    return df[['symbol', 'date', 'open', 'high', 'low', 'close', 'volume']]

skipped = []
frames = []

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_map = {executor.submit(fetch_one, sym): sym for sym in STOCK_LIST}
    for future in concurrent.futures.as_completed(future_map):
        sym = future_map[future]
        try:
            df = future.result()
            print(f'  [OK] {sym:20s} {len(df)} rows')
            frames.append(df)
        except Exception as e:
            print(f'  [SKIP] {sym:20s} {e}')
            skipped.append(sym)

if skipped:
    print(f'\nSkipped: {skipped}')

combined = pd.concat(frames, ignore_index=True)
combined.sort_values(['symbol', 'date'], inplace=True)
combined.reset_index(drop=True, inplace=True)
print(f'[OK] Combined: {len(combined)} rows, {combined["symbol"].nunique()} stocks')

## 4. Compute Metrics

In [ ]:
print('Computing metrics...')

hl_rows = []
cross_rows = []
macd_rows = []
return_rows = []

for sym, grp in combined.groupby('symbol'):
    grp = grp.sort_values('date').reset_index(drop=True)
    
    try:
        hl = compute_period_hl(grp)
        hl['Symbol'] = sym
        hl_rows.append(hl)
    except Exception as e:
        print(f'  Warning [{sym}] HL: {e}')
    
    try:
        if len(grp) >= 5:
            for c in compute_sma_crossovers(grp):
                c['Symbol'] = sym
                cross_rows.append(c)
    except Exception as e:
        print(f'  Warning [{sym}] SMA: {e}')
    
    try:
        macd = compute_macd(grp)
        macd['Symbol'] = sym
        macd_rows.append(macd)
    except Exception as e:
        print(f'  Warning [{sym}] MACD: {e}')
    
    try:
        ret = compute_period_returns(grp)
        if ret:
            ret['Symbol'] = sym
            return_rows.append(ret)
    except Exception as e:
        print(f'  Warning [{sym}] Returns: {e}')

hl_df = pd.DataFrame(hl_rows)
cross_df = pd.DataFrame(cross_rows) if cross_rows else pd.DataFrame()
macd_df = pd.DataFrame(macd_rows)
returns_df = pd.DataFrame(return_rows)

print(f'[OK] Metrics: {len(hl_df)} HL, {len(cross_df)} crossovers, {len(macd_df)} MACD, {len(returns_df)} returns')

## 5. Generate Reports

In [ ]:
# Report 1: 52W Position
print('\n[REPORT 1] 52W Hi/Low Position (Top 20)')
r1 = hl_df[['Symbol', 'Current_Price', '52W_High', '52W_Low', '52W_Position']].sort_values('52W_Position')
print(r1.head(20).to_string(index=False))

In [ ]:
# Report 2: Crossovers
print('\n[REPORT 2] Golden Crossover Signals')
crossed = pd.DataFrame()
nearing = pd.DataFrame()

if not cross_df.empty:
    crossed = cross_df[cross_df['crossed_last_5d'] == True][['Symbol', 'cross_type', 'cross_pct', 'last_cross_date']]
    if not crossed.empty:
        print('\n[OK] Crossed in last 5 days:')
        print(crossed.to_string(index=False))
    
    nearing = cross_df[cross_df['nearing'] == True][['Symbol', 'cross_type', 'cross_pct']]
    if not nearing.empty:
        print('\n[OK] Nearing crossover:')
        print(nearing.to_string(index=False))
else:
    print('  (no crossover data)')

In [ ]:
# Report 3: MACD
print('\n[REPORT 3] MACD Signals (Score >= 2)')
r3 = macd_df[macd_df['MACD_Score'] >= 2].sort_values('MACD_Score', ascending=False)
r3_display = pd.DataFrame()

if not r3.empty:
    r3_display = r3[['Symbol', 'MACD', 'Signal', 'Histogram', 'MACD_Score']]
    print(r3_display.to_string(index=False))
else:
    print('  (no stocks with MACD score >= 2)')

In [ ]:
# Report 4: Near 52W High/Low
print('\n[REPORT 4] Near 52W High / Low')
near_high = hl_df[hl_df['52W_Position'] >= 80].sort_values('52W_Position', ascending=False).head(10)
near_low = hl_df[hl_df['52W_Position'] <= 20].sort_values('52W_Position').head(10)

if not near_high.empty:
    print('\n[OK] Near 52W HIGH (>=80%):')
    print(near_high[['Symbol', 'Current_Price', '52W_Position']].to_string(index=False))

if not near_low.empty:
    print('\n[OK] Near 52W LOW (<=20%):')
    print(near_low[['Symbol', 'Current_Price', '52W_Position']].to_string(index=False))

In [ ]:
# Report 5: Top Gainers/Losers
print('\n[REPORT 5] Top Movers (1 Month)')
gainers = returns_df[returns_df['1M_%'] > 0].nlargest(10, '1M_%')[['Symbol', 'Current_Price', '1M_%']]
losers = returns_df[returns_df['1M_%'] < 0].nsmallest(10, '1M_%')[['Symbol', 'Current_Price', '1M_%']]

if not gainers.empty:
    print('\n[+] TOP 10 GAINERS (1M):')
    print(gainers.to_string(index=False))

if not losers.empty:
    print('\n[-] TOP 10 LOSERS (1M):')
    print(losers.to_string(index=False))

## 6. Generate Charts

Note: Chart generation code is available in the Python script version (`nse_analysis.py`). Run the script to generate charts and send email with embedded visualizations.

## 7. Save Results

In [ ]:
timestamp = datetime.now().strftime('%Y-%m-%d-%H-%M-%S')
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = f"{OUTPUT_DIR}/NSE-ANALYSIS-{timestamp}.csv"

csv_sections = [
    pd.DataFrame([['=== REPORT 1: 52W Hi/Low Position ===']]), r1, pd.DataFrame([[]]),
    pd.DataFrame([['=== REPORT 2A: Crossed Last 5 Days ===']]),
    crossed if not crossed.empty else pd.DataFrame([['(none)']]),
    pd.DataFrame([[]]),
    pd.DataFrame([['=== REPORT 2B: Nearing Crossover ===']]),
    nearing if not nearing.empty else pd.DataFrame([['(none)']]),
    pd.DataFrame([[]]),
    pd.DataFrame([['=== REPORT 3: MACD Signals ===']]),
    r3_display if not r3_display.empty else pd.DataFrame([['(none)']]),
    pd.DataFrame([[]]),
    pd.DataFrame([['=== REPORT 4A: Near 52W HIGH ===']]),
    near_high if not near_high.empty else pd.DataFrame([['(none)']]),
    pd.DataFrame([[]]),
    pd.DataFrame([['=== REPORT 4B: Near 52W LOW ===']]),
    near_low if not near_low.empty else pd.DataFrame([['(none)']]),
]

with open(out_path, 'w', newline='') as f:
    for section in csv_sections:
        section.to_csv(f, index=False, header=True)
        f.write('\n')

print(f'\n[OK] Reports saved to: {out_path}')
if skipped:
    print(f'[WARNING] Skipped stocks: {skipped}')

## Complete

For full functionality including chart generation and email sending, run the Python script version:

```bash
python nse_analysis.py
```